# Week 4: Transfer learning — guided seminar

This notebook is an ungraded, complete example.
We adapt a pretrained BERT encoder to named entity recognition (NER).
The homework uses the same tools on a different task: duplicate-question detection.
You may reuse this seminar's code in your homework.
Submit only the homework, with its experiment report; there is no seminar submission.

The transfer-learning recipe from the lecture, as it appears here:

- **Source task:** masked language modeling on unlabeled text.
  The BERT authors have done this step; we download their checkpoint.
- **Target task:** NER, with labels for about 15% of the CoNLL-2003 training sentences.
- **Adaptation:** a new task head on top of the pretrained encoder, then fine-tuning of the whole model.

| Section | Lecture slides |
| --- | --- |
| 1. Tokens, padding and attention masks | BERT: input representations |
| 2. A pretrained encoder and a task head | BERT: a pretrained Transformer encoder; BERT: masked language modeling |
| 3. NER data | BERT: token-level tasks |
| 4–5. Trainable parameters and fine-tuning | So far: a new head for every shape of output; Four ways to use one pretrained model |
| 6. What did pretraining contribute? | Transfer learning |
| 7. A short connection to T5 | Generation reuses the pretrained output layer |

By the end, explain where pretrained weights enter the model,
how word labels become token labels, and which parameters receive gradients.
Each section starts with the idea it demonstrates;
a **What to notice** cell after an output says what to look at.

In [ ]:
# Colab already provides PyTorch, Transformers 5 and Datasets.
# Elsewhere, install PyTorch for your CPU/GPU, then transformers>=5.18,<6 and datasets>=3,<5.
import gc
import time
from collections import Counter

import datasets
import torch
import transformers
from torch.utils.data import DataLoader
from transformers import (
    AutoConfig,
    AutoModelForMaskedLM,
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorForTokenClassification,
    pipeline,
    set_seed,
)

SEED = 42
BASE_MODEL = "google-bert/bert-base-cased"
MODEL_REVISION = "cd5ef92a9fb2f889e972770a36d4ed042daf221e"
DATASET_REVISION = "19edcb426bfd625c275c17b9a99b2239243f4377"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
set_seed(SEED)
print("Versions:", torch.__version__, transformers.__version__, datasets.__version__)
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(DEVICE))

## 1. Tokens, padding and attention masks

The tokenizer is part of the pretrained checkpoint.
Each token ID selects a row of BERT's pretrained embedding matrix,
so a model must be used with the tokenizer it was pretrained with.

BERT splits text into WordPiece subwords from a fixed vocabulary.
A rare word becomes several pieces instead of an unknown-word token;
`##` marks a piece that continues the previous one.
We use the **cased** model: capitalization is a strong cue for names.

A batch must be a rectangular tensor, so shorter sequences are padded.
Dynamic padding pads only to the longest sequence in the current batch.
`attention_mask` marks real positions with 1 and padding with 0,
so no token attends to padding.
It is not the supervision mask used in the loss; section 3 compares the two.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL, revision=MODEL_REVISION, use_fast=True
)
batch = tokenizer(
    ["Michael studies transfer learning.", "An electroencephalographic recording."],
    padding=True,
    return_tensors="pt",
)
print({key: tuple(value.shape) for key, value in batch.items()})
for row, mask in zip(batch["input_ids"], batch["attention_mask"]):
    print(list(zip(tokenizer.convert_ids_to_tokens(row.tolist()), mask.tolist())))

**What to notice.**
Common words stay whole; "electroencephalographic" becomes five pieces.
Both rows have the same length:
the shorter sentence ends with `[PAD]` positions whose mask is 0.
The tokenizer adds `[CLS]` and `[SEP]` itself; they are not in the text.

BERT can also read two texts at once, as the homework does with question pairs.
The input vector of every token is the sum of three learned embeddings:
token, segment (`token_type_ids`: first or second text) and position.
BERT has 512 learned positions, so longer inputs must be truncated.

In [ ]:
pair = tokenizer("Can an encoder learn useful features?", "Yes, from pretraining.")
print(tokenizer.convert_ids_to_tokens(pair["input_ids"]))
print("Segments:", pair["token_type_ids"])

**What to notice.**
One `[CLS]` starts the input, and a `[SEP]` closes each text.
Segment IDs switch from 0 to 1 after the first `[SEP]`.
In the homework, a classifier reads the final `[CLS]` vector of such a pair.

## 2. A pretrained encoder and a task head

A pretrained model splits into a **body** and a **head**.
The body, BERT's 12-layer encoder, returns one contextual vector per token:
`[batch, sequence, hidden]`.
Unlike word2vec or GloVe vectors, each one depends on the whole input, on both sides.

The head turns those vectors into a task's outputs.
The masked-language-model head maps each vector to scores over the vocabulary:
`[batch, sequence, vocabulary]`.
For NER we will keep the body and replace this head with a new one over nine entity tags.

Transformers prints a load report for the checkpoint.
The pooler and the next-sentence-prediction head are `UNEXPECTED`.
BERT was pretrained with two objectives, masked tokens and next-sentence prediction,
and the checkpoint keeps the parts for both.
The MLM model uses only the first; later work found next-sentence prediction unnecessary.

In [ ]:
mlm = AutoModelForMaskedLM.from_pretrained(BASE_MODEL, revision=MODEL_REVISION)
mlm.eval()
with torch.no_grad():
    representations = mlm.base_model(**batch).last_hidden_state
    vocabulary_logits = mlm(**batch).logits
print("Encoder output:", tuple(representations.shape))
print("MLM logits:", tuple(vocabulary_logits.shape))

**What to notice.**
The body gives 768 numbers per token.
The MLM head turns them into 28,996 scores per token, one per entry of the cased vocabulary.
Any head that accepts 768-dimensional vectors can sit on the same body.

Masked language modeling is the pretraining objective:
hide some input tokens and predict the originals from the context on both sides.
Filling a blank well takes syntax, meaning and some knowledge of the world,
yet the training text needs no labels.
Below, the same sentence frame gets two different subjects.

In [ ]:
fill_mask = pipeline("fill-mask", model=mlm, tokenizer=tokenizer, device=-1)
for text in ["The doctor examined the [MASK].", "The mechanic examined the [MASK]."]:
    print(text)
    print([(item["token_str"], round(item["score"], 3)) for item in fill_mask(text)])

**What to notice.**
The output is a probability distribution, not one answer;
even the top word gets only a few percent.
Changing one word on the left moves probability toward words that fit that context.
This illustrates the objective;
a handful of completions is not evidence of what the model knows.

In [ ]:
del fill_mask, mlm, representations, vocabulary_logits
gc.collect()

## 3. NER data: words are not model tokens

CoNLL-2003 consists of Reuters news sentences annotated with four entity types:
persons (`PER`), organizations (`ORG`), locations (`LOC`) and miscellaneous names (`MISC`).
Tags follow IOB2: `B-X` begins an entity of type X, `I-X` continues it, `O` is outside any entity.
`B-` lets two adjacent entities of the same type stay separate.

We take 2,048 of the 14,041 training sentences, about 15%, and 512 validation sentences.
Fixed revisions, a seed and fixed subsets make the data and initialization repeatable;
GPU training can still vary slightly between runs.
These sizes and the epoch count are teaching defaults, not a measured runtime budget.
Use a GPU if available; CPU execution of BERT is slow.
The official test split is left untouched.

In [ ]:
raw = datasets.load_dataset("lhoestq/conll2003", revision=DATASET_REVISION)
label_names = raw["train"].features["ner_tags"].feature.names
id2label = dict(enumerate(label_names))
label2id = {label: index for index, label in id2label.items()}
train_words = raw["train"].shuffle(seed=SEED).select(range(2048))
dev_words = raw["validation"].shuffle(seed=SEED).select(range(512))
print("Labels:", label_names)
print(list(zip(train_words[0]["tokens"], train_words[0]["ner_tags"])))

**What to notice.**
Tags are integers that index `label_names`; most words carry 0, which is `O`.
The tags belong to words as the annotators split them, not to BERT's tokens.

The model, however, outputs one vector per subword token.
Following the BERT paper, we supervise only the **first subword** of each word.
Other subwords, special tokens and padding receive label `-100`,
the index that PyTorch cross-entropy ignores.
Another valid policy labels all subwords, but then continuation pieces need `I-` tags.

Two different masks now describe each position:

| Position | `attention_mask` | Label |
| --- | --- | --- |
| Padding | 0 | `-100` |
| `[CLS]`, `[SEP]` | 1 | `-100` |
| Second and later pieces of a word | 1 | `-100` |
| First piece of a word | 1 | the word's tag |

`attention_mask` controls what attention can see; `-100` controls what the loss counts.
A position ignored by the loss still informs the vectors of all other tokens.

In [ ]:
def align_word_labels(word_ids, word_labels):
    aligned = []
    previous_word = None
    for word_id in word_ids:
        if word_id is None or word_id == previous_word:
            aligned.append(-100)
        else:
            aligned.append(word_labels[word_id])
        previous_word = word_id
    return aligned


def tokenize_and_align(samples):
    encoded = tokenizer(
        samples["tokens"],
        is_split_into_words=True,
        truncation=True,
        max_length=128,
    )
    encoded["labels"] = [
        align_word_labels(encoded.word_ids(batch_index=i), labels)
        for i, labels in enumerate(samples["ner_tags"])
    ]
    return encoded


assert align_word_labels([None, 0, 1, 1, 2, None], [1, 0, 5]) == [-100, 1, 0, -100, 5, -100]
example = tokenize_and_align(train_words[:1])
print(list(zip(
    tokenizer.convert_ids_to_tokens(example["input_ids"][0]),
    example["labels"][0],
)))

**What to notice.**
Each word's tag appears once, at its first piece;
continuation pieces such as `##s` get `-100`.
`[CLS]` and `[SEP]` also get `-100`: they are not words.

In [ ]:
train_tokens = train_words.map(
    tokenize_and_align, batched=True, remove_columns=train_words.column_names
)
dev_tokens = dev_words.map(
    tokenize_and_align, batched=True, remove_columns=dev_words.column_names
)
collator = DataCollatorForTokenClassification(tokenizer, label_pad_token_id=-100)
train_loader = DataLoader(
    train_tokens, batch_size=16, shuffle=True, collate_fn=collator,
    generator=torch.Generator().manual_seed(SEED),
)
dev_loader = DataLoader(dev_tokens, batch_size=16, shuffle=False, collate_fn=collator)
sample_batch = next(iter(train_loader))
print({key: tuple(value.shape) for key, value in sample_batch.items()})
print("Attention mask:", sample_batch["attention_mask"][0].tolist())
print("Loss labels:", sample_batch["labels"][0].tolist())

**What to notice.**
All tensors in a batch share one length, set by the longest sentence in it.
At padding positions the mask is 0 and the label is `-100`.
Inside the sentence the mask is 1 everywhere,
while the labels skip special tokens and continuation pieces.

## 4. Which parameters learn?

The body is pretrained; the head is new.
`AutoModelForTokenClassification` adds a linear layer that maps each 768-dimensional token vector
to nine tag scores, and initializes it randomly.
In the load report, the classifier is `MISSING`: its weights are not in the checkpoint.
Pretraining heads this model does not use are `UNEXPECTED`.

The lecture lists two ways to adapt such a model:
keep the body **frozen** and train only the head, or **fine-tune** both.
Below we freeze the body, run one backward pass and check where gradients arrive.
There is no optimizer step here, so the weights do not change.

Three switches are easy to confuse:

| Switch | Controls | Does not |
| --- | --- | --- |
| `requires_grad_(False)` | whether a parameter receives gradients | skip the forward pass |
| `model.eval()` / `model.train()` | dropout and other training-only behavior | stop gradients |
| `torch.no_grad()` | whether autograd records the computation | change dropout |

A frozen encoder still runs forward on every batch;
freezing saves its backward pass and its optimizer state.

In [ ]:
set_seed(SEED)
model = AutoModelForTokenClassification.from_pretrained(
    BASE_MODEL,
    revision=MODEL_REVISION,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id,
).to(DEVICE)


def parameter_counts(model):
    return {
        "total": sum(parameter.numel() for parameter in model.parameters()),
        "trainable": sum(
            parameter.numel() for parameter in model.parameters() if parameter.requires_grad
        ),
    }


for parameter in model.base_model.parameters():
    parameter.requires_grad_(False)
model.train()
model.base_model.eval()
device_batch = {key: value.to(DEVICE) for key, value in sample_batch.items()}
model(**device_batch).loss.backward()
print("Frozen:", parameter_counts(model))
print("Encoder has gradients:", any(p.grad is not None for p in model.base_model.parameters()))
print("Head has gradients:", any(p.grad is not None for p in model.classifier.parameters()))
assert all(p.grad is None for p in model.base_model.parameters())
assert any(p.grad is not None for p in model.classifier.parameters())
model.zero_grad(set_to_none=True)
for parameter in model.base_model.parameters():
    parameter.requires_grad_(True)
print("Full fine-tuning:", parameter_counts(model))

**What to notice.**
With a frozen body, 6,921 of about 107.7 million parameters are trainable:
768 × 9 head weights plus 9 biases.
All pretrained knowledge sits in the frozen part.
Full fine-tuning makes every parameter trainable.
AdamW then keeps two running averages per parameter in addition to its gradient,
so training needs several times the memory of the weights alone.

In the BERT paper, on the full CoNLL-2003 data, frozen features came close to fine-tuning:
96.1 vs 96.4 dev F1 for BERT-base.
There the frozen features fed a trained two-layer BiLSTM, not a single linear layer,
and the model read whole documents rather than single sentences.
The homework measures the gap on another task, with one linear head for both methods.

## 5. One complete fine-tuning run

**Metric.** Most words are `O`,
so token accuracy stays high even for a model that finds no entities.
We use entity-level precision, recall and F1 with strict IOB2 matching.
An entity is `B-X` followed by any number of `I-X` tags.
A prediction counts only if its type and both boundaries match a gold entity.
An `I-X` tag that does not continue an entity of type X starts no entity.
A boundary error therefore costs twice: one false positive and one missed gold entity.
Positions labelled `-100` are dropped before scoring, so the metric works on words, not subwords.

Before training the head is random, so the scores should be near zero.

In [ ]:
def entity_spans(tags):
    spans, start, kind = set(), None, None
    for position, tag in enumerate(tags + ["O"]):
        if start is not None and tag != f"I-{kind}":
            spans.add((kind, start, position))
            start = None
        if tag.startswith("B-"):
            start, kind = position, tag[2:]
    return spans


def entity_scores(references, predictions):
    gold_count = predicted_count = correct = 0
    for reference, prediction in zip(references, predictions):
        gold, predicted = entity_spans(reference), entity_spans(prediction)
        gold_count += len(gold)
        predicted_count += len(predicted)
        correct += len(gold & predicted)
    precision = correct / predicted_count if predicted_count else 0.0
    recall = correct / gold_count if gold_count else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"precision": precision, "recall": recall, "f1": f1}


assert entity_spans(["B-PER", "I-PER", "O", "I-LOC", "B-ORG", "I-LOC"]) == {("PER", 0, 2), ("ORG", 4, 5)}
assert entity_scores([["B-PER", "I-PER", "B-LOC"]], [["B-PER", "O", "B-LOC"]]) == {
    "precision": 0.5, "recall": 0.5, "f1": 0.5,
}


@torch.no_grad()
def predict_word_tags(model, loader):
    model.eval()
    references, predictions = [], []
    for batch in loader:
        labels = batch["labels"]
        inputs = {key: value.to(DEVICE) for key, value in batch.items() if key != "labels"}
        predicted = model(**inputs).logits.argmax(dim=-1).cpu()
        for predicted_row, label_row in zip(predicted.tolist(), labels.tolist()):
            kept = [(prediction, label) for prediction, label in zip(predicted_row, label_row) if label != -100]
            predictions.append([id2label[prediction] for prediction, _ in kept])
            references.append([id2label[label] for _, label in kept])
    return references, predictions


def evaluate_ner(model, loader):
    return entity_scores(*predict_word_tags(model, loader))


print("Before training:", evaluate_ner(model, dev_loader))

**Training.** Learning rate 3e-5, batch size 16 and 3 epochs lie within the fine-tuning ranges
recommended in the BERT paper (Appendix A.3).
The learning rate is small because all pretrained parameters move:
large steps could overwrite what pretraining learned before the new head becomes useful.
Training only a head usually uses a much larger rate; the homework's frozen run starts at 1e-3.
Gradient clipping bounds the size of each update.
We check the validation subset after every epoch and never tune on the test set.

`train_ner` resets the seed and the data order, so section 6 can repeat the run exactly.

In [ ]:
EPOCHS = 3
LEARNING_RATE = 3e-5


def train_ner(model):
    set_seed(SEED)
    train_loader.generator.manual_seed(SEED)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    started = time.perf_counter()
    for epoch in range(EPOCHS):
        model.train()
        total_loss, batches = 0.0, 0
        for batch in train_loader:
            inputs = {key: value.to(DEVICE) for key, value in batch.items()}
            optimizer.zero_grad(set_to_none=True)
            loss = model(**inputs).loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_loss += loss.item()
            batches += 1
        print({"epoch": epoch + 1, "train_loss": total_loss / batches, **evaluate_ner(model, dev_loader)})
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    print("Training + validation seconds:", time.perf_counter() - started)


train_ner(model)

**What to notice.**
Dev F1 is already high after the first epoch, only 128 updates on 2,048 sentences:
the pretrained body supplies most of what NER needs.
Training loss keeps falling faster than dev F1 rises;
the model is starting to fit the training sentences themselves.
With about 900 dev entities, a difference of about one F1 point is close to noise.
We looked at dev scores while choosing this setup, so dev F1 is slightly optimistic;
a final number would come from the untouched test split.
This short run demonstrates adaptation; it is not BERT's attainable NER quality.

At inference time there are no labels to mark the first subword of each word.
Use `word_ids()` to map token predictions back to words;
this is also how you would tag new, unlabeled text.
The columns below are the word, its gold tag and the predicted tag.

In [ ]:
model.eval()
sample = dev_words[0]
encoded = tokenizer(sample["tokens"], is_split_into_words=True, return_tensors="pt", truncation=True, max_length=128)
with torch.no_grad():
    predicted = model(**{key: value.to(DEVICE) for key, value in encoded.items()}).logits.argmax(-1)[0].cpu().tolist()
previous_word = None
for token_position, word_id in enumerate(encoded.word_ids()):
    if word_id is not None and word_id != previous_word:
        print(sample["tokens"][word_id], id2label[sample["ner_tags"][word_id]], id2label[predicted[token_position]])
    previous_word = word_id

One sentence rarely contains every kind of error, so classify errors over the whole dev subset.
For each gold entity without an exact match:
a predicted entity with the same boundaries but another type is a **wrong type**;
an overlapping prediction with different boundaries is a **boundary** error;
no overlapping prediction means the model **missed** the entity.
Predicted entities that overlap no gold entity are false positives; this cell does not list them.
Discuss one example of each category.
After a short run a category can still be absent; do not invent examples.

In [ ]:
def unmatched_gold_entities(reference, prediction):
    predicted = sorted(entity_spans(prediction), key=lambda span: span[1])
    for kind, start, end in sorted(entity_spans(reference) - set(predicted), key=lambda span: span[1]):
        same_boundaries = [span for span in predicted if span[1:] == (start, end)]
        overlapping = [span for span in predicted if span[1] < end and start < span[2]]
        if same_boundaries:
            yield "wrong type", (kind, start, end), same_boundaries
        elif overlapping:
            yield "boundary", (kind, start, end), overlapping
        else:
            yield "missed", (kind, start, end), []


def span_text(words, span):
    kind, start, end = span
    return f"{kind} '{' '.join(words[start:end])}'"


dev_references, dev_predictions = predict_word_tags(model, dev_loader)
counts, examples = Counter(), {}
for index, (reference, prediction) in enumerate(zip(dev_references, dev_predictions)):
    for category, gold, predicted in unmatched_gold_entities(reference, prediction):
        counts[category] += 1
        examples.setdefault(category, (index, gold, predicted))
print("Gold entities:", sum(len(entity_spans(reference)) for reference in dev_references))
print("Without an exact match:", {category: counts[category] for category in ("missed", "wrong type", "boundary")})
for category, (index, gold, predicted) in examples.items():
    words = dev_words[index]["tokens"]
    found = ", ".join(span_text(words, span) for span in predicted) or "nothing"
    print(f"\n{category}: gold {span_text(words, gold)}, predicted {found}")
    print("   ", " ".join(words))

**What to notice.**
Wrong types often involve names whose type depends on the context or on annotation conventions;
`MISC` collects names that fit no other type.
Reuters text has its own conventions, such as all-caps headlines and datelines.
The gold tags are not perfect either:
one study found label mistakes in about 5% of CoNLL-2003 test sentences
([Wang et al., 2019](https://arxiv.org/abs/1909.01441)).
Some apparent errors may be disagreements with the annotation rather than misreadings of the text.

## 6. What did pretraining contribute?

Transfer learning claims that pretraining on unlabeled text makes labeled data go further.
To see this directly, build the same architecture with **random** weights
and train it with the same data, recipe and seed.
`from_config` creates a model from its description without loading pretrained weights.

This is not a tuned baseline:
a model trained from scratch would need its own recipe and usually far more labeled data.
The comparison changes one thing, the initialization of the body.

In [ ]:
set_seed(SEED)
scratch_config = AutoConfig.from_pretrained(
    BASE_MODEL,
    revision=MODEL_REVISION,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id,
)
scratch_model = AutoModelForTokenClassification.from_config(scratch_config).to(DEVICE)
print("Random initialization:", parameter_counts(scratch_model))
train_ner(scratch_model)

**What to notice.**
Both models have the same parameters; only their starting values differ.
Compare dev F1 epoch by epoch with section 5.
The gap is what pretraining contributed under these conditions:
2,048 labeled sentences and 384 updates.

## 7. A short connection to T5

Our NER model answers through a new head with a fixed set of nine tags.
T5 writes every answer as text, through an output layer that was itself pretrained.
Its pretraining objective is also denoising, but over spans:
T5 gives the encoder a corrupted sequence and trains the decoder to generate the missing spans.
Each span has its own sentinel, and the target ends with another sentinel.
Both encoder input and target end with EOS.
This word-level toy example illustrates the sequence layout;
a real T5 tokenizer operates on subword IDs, not these space-separated words.
We do not load or train another model here.

In [ ]:
source_tokens = ["The", "small", "cat", "sat", "on", "the", "warm", "mat", "."]
encoder_tokens = ["The", "<extra_id_0>", "sat", "on", "the", "<extra_id_1>", ".", "</s>"]
target_tokens = ["<extra_id_0>", "small", "cat", "<extra_id_1>", "warm", "mat", "<extra_id_2>", "</s>"]
print("Original:", " ".join(source_tokens))
print("Encoder input:", " ".join(encoder_tokens))
print("Target:", " ".join(target_tokens))

Teacher forcing gives the decoder previous **gold** target tokens during training.
Its self-attention is causal; cross-attention can access the encoded input.
For T5, the decoder starts with the pad token, then receives the shifted target.
Loss covers target sentinels, missing words and EOS; padding labels are ignored.
The start token is a real decoder position even though its ID equals the pad ID.

In [ ]:
toy_vocabulary = {"<pad>": 0, "</s>": 1}
for token in encoder_tokens + target_tokens:
    if token not in toy_vocabulary:
        toy_vocabulary[token] = len(toy_vocabulary)
labels = [toy_vocabulary[token] for token in target_tokens] + [-100, -100]
decoder_input_ids = [toy_vocabulary["<pad>"]] + [
    token_id if token_id != -100 else toy_vocabulary["<pad>"]
    for token_id in labels[:-1]
]
print("Toy labels:", labels)
print("Shifted decoder inputs:", decoder_input_ids)
print("Positions contributing to loss:", [label != -100 for label in labels])
assert len(labels) == len(decoder_input_ids)

## Check yourself

These questions are not graded; each answer is in a section above.

1. Why must a pretrained model be used with its own tokenizer?
2. How do `attention_mask` and label `-100` differ? Name a position where one is 1 and the other is `-100`.
3. Which parameters were pretrained, which were new, and how many receive gradients in each regime?
4. What does each of `requires_grad_(False)`, `model.eval()` and `torch.no_grad()` change?
5. Why is entity-level F1 more informative than token accuracy for NER, and why does a boundary error cost twice?
6. What differed between the pretrained and the randomly initialized runs, and what stayed the same?

## Take this to the homework

Reuse tokenization, dynamic padding, device handling, gradient inspection and the training loop.
Replace token labels with one binary label per question pair.
Compare a frozen encoder plus a trained head with full fine-tuning,
starting both runs from the same pretrained encoder and the same initial head.
There is one homework submission worth 9 required points; the seminar is practice.

References: [HF token-classification guide](https://huggingface.co/docs/transformers/en/tasks/token_classification),
[BERT](https://arxiv.org/abs/1810.04805), [T5](https://jmlr.org/papers/v21/20-074.html),
[CrossWeigh](https://arxiv.org/abs/1909.01441).